In [ ]:
import os
import sys
import subprocess
from pathlib import Path

os.environ['PATH'] = f"/root/.cargo/bin:{os.environ['PATH']}"

ENDWITHS = 'page-level'

NOTEBOOK_DIR = os.getcwd()

if not NOTEBOOK_DIR.endswith(ENDWITHS):
    raise ValueError(f"Not in correct dir, expect end with {ENDWITHS}, but got {NOTEBOOK_DIR} instead")

BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
print(BASE_DIR)

sys.path.insert(0, os.path.join(BASE_DIR))


# LOAD CREDENTIALS

## For Local

In [ ]:
from dotenv import load_dotenv
load_dotenv(os.path.abspath(os.path.join(BASE_DIR,'..','.env')))

HF_TOKEN = os.getenv('HF_TOKEN')
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')

# CONSTANT VARIABLES, change this!

user/base-model-layer-module-rank
layer = l12, l8, 
module = down , all
rank = r1 r32

In [ ]:
from finetune_utils.antoanai_util import *

In [ ]:
MODEL = "unsloth/Qwen2.5-0.5B-Instruct" # Model from Hugging Face to train, change this
TRAINING_FILE_NAME = 'train_full_context.jsonl' # change this
LAYERS_TO_TRANSFORM = None # Put in  [ ], the layers you want to transform IF None, all layers are transformed
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj"
]
# Avaiable: ["down_proj"]  ||||  ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"] 
RANK = 32 # Rank of the Lora
# --- ADD THIS LINE ---
FULL_FINETUNE = True # Set to True for full finetuning (non-LoRA), False for LoRA
# ---------------------
TRAINING_DIR = os.path.join(BASE_DIR, 'data', 'translation', 'bsd_ja_en', 'training')
TRAINING_FILE = os.path.join(TRAINING_DIR, TRAINING_FILE_NAME) # change this

CONFIG_DIR = os.path.join(BASE_DIR, 'data', 'translation', 'bsd_ja_en', 'configs')

LORA_ALPHA = 64 # Lora alpha
LEARNING_RATE = 1e-5 # Learning rate
EPOCHS = 1 # Training epochs

FINETUNED_MODEL_ID = f"TheBlindMaster/{create_model_id(MODEL, TRAINING_FILE_NAME, TARGET_MODULES, LAYERS_TO_TRANSFORM, RANK, EPOCHS)}"
CONFIG_FILE = os.path.join(CONFIG_DIR, f"{create_model_id(MODEL, TRAINING_FILE_NAME, TARGET_MODULES, LAYERS_TO_TRANSFORM, RANK, EPOCHS)}.json")

OVERWRITE_EXISTING_CONFIG_FILE = False
DELETE_MODEL_AFTER_TRAINING = False

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(f"{MODEL}")  # Replace with your model

file_path = f"{TRAINING_FILE}"

max_sample_length = 0

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        tokens = tokenizer.encode(line, add_special_tokens=True)
        length = len(tokens)
        if length > max_sample_length:
            max_sample_length = length
            longest_line = line

print(f"Max tokenized sequence length: {max_sample_length}")

In [ ]:
import json

# Changing the config further to your needs
model_config = {
    "model": f"{MODEL}",
    "training_file": f"{TRAINING_FILE}",
    "finetuned_model_id": f"{FINETUNED_MODEL_ID}",
    "max_seq_length": max_sample_length + 256,  # Adding some buffer for special tokens
    "loss": "sft",
    "target_modules": TARGET_MODULES,
    "layers_to_transform": LAYERS_TO_TRANSFORM,
    "r": RANK,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "warmup_steps": 5,
    "optim": "adamw_8bit",
    "epochs": EPOCHS,
    "push_to_private": True,
    "merge_before_push": True,
    "train_on_responses_only": True,
    "save_steps": 100
}

In [ ]:
os.makedirs(CONFIG_DIR, exist_ok=True)

if os.path.exists(CONFIG_FILE) and not OVERWRITE_EXISTING_CONFIG_FILE:
    raise FileExistsError(f"Config file {CONFIG_FILE} already exists. Set OVERWRITE_EXISTING_CONFIG_FILE=True to overwrite.")

In [ ]:
with open(CONFIG_FILE, "w") as f:
    json.dump(model_config, f, indent=2)

print("Lightweight config created!")
print("Remember to change 'finetuned_model_id' to your HuggingFace username!")
print(f"Config file saved at: {CONFIG_FILE}")

# Add to path and update global variables
import sys
sys.path.insert(0, BASE_DIR)

import os
os.chdir(BASE_DIR)
print(f"Changed to: {os.getcwd()}")

In [ ]:
!cat "{CONFIG_FILE}"

In [ ]:
# Execute fine-tuning
# MAKE SURE TO USE THE SAME PYTHON
script_name = "run_full_finetune.py" if FULL_FINETUNE else "run_finetune.py"

print(f"Run this in terminal with the same interpreter or in a Jupyter Notebook cell: \n \n python3 {BASE_DIR}/src/bubble-translation/page-level/finetune_utils/{script_name} {CONFIG_FILE}")

# QUEUE Finetuning

In [ ]:
print(f"python3 {BASE_DIR}/src/bubble-translation/page-level/finetune_utils/run_finetune_batch.py {CONFIG_DIR} {DELETE_MODEL_AFTER_TRAINING}")